## Backend de RAG

### Importar librerias y utilidades

In [7]:
import io
import os
from dotenv import load_dotenv
load_dotenv()
from azure.core.credentials import AzureKeyCredential
from azure.storage.blob import BlobServiceClient
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeResult
from openai import AzureOpenAI
import requests


### Importar variables de entorno y variables utilies para el procesamiento

In [2]:
AZURE_STORAGE_CONNECTION_STRING = os.getenv("AZURE_STORAGE_CONNECTION_STRING")
DEPLOYMENT = "phi-4-mini-instruct"
DOCUMENT_INTELLIGENCE_ENDPOINT = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
DOCUMENT_INTELLIGENCE_KEY = os.getenv("DOCUMENT_INTELLIGENCE_KEY")
client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_version="2024-08-01-preview"  
)

#### Funcion para obtener texto de pdf 

In [3]:
def obtener_texto_de_blob_pdf(BLOB_NAME, CONTAINER_NAME):
    try:

        if not all([AZURE_STORAGE_CONNECTION_STRING, DOCUMENT_INTELLIGENCE_ENDPOINT, DOCUMENT_INTELLIGENCE_KEY]):
            raise ValueError("Faltan configurar variables en tu archivo .env. Por favor verifícalo.")

        print(f"Conectando a Blob Storage para descargar: {BLOB_NAME}...")
        blob_service_client = BlobServiceClient.from_connection_string(AZURE_STORAGE_CONNECTION_STRING)
        blob_client = blob_service_client.get_blob_client(container=CONTAINER_NAME, blob=BLOB_NAME)
        

        pdf_bytes = blob_client.download_blob().readall()
        
    
        pdf_stream = io.BytesIO(pdf_bytes)
        
        print("Enviando el archivo a Azure AI Document Intelligence...")
        client = DocumentIntelligenceClient(
            endpoint=DOCUMENT_INTELLIGENCE_ENDPOINT, 
            credential=AzureKeyCredential(DOCUMENT_INTELLIGENCE_KEY)
        )
        
        poller = client.begin_analyze_document(
            model_id="prebuilt-layout",
            body=pdf_stream,
            content_type="application/pdf",
            output_content_format="markdown"
        )
        
        resultado: AnalyzeResult = poller.result()
        texto_final_string = resultado.content
        return texto_final_string

    except Exception as e:
        print(f"Ocurrió un error en el proceso: {e}")
        return None

In [4]:
text_pdf = obtener_texto_de_blob_pdf('7245fcea1c7bf0a995952dabe3da456d.pdf','pdf')
print(text_pdf)

Conectando a Blob Storage para descargar: 7245fcea1c7bf0a995952dabe3da456d.pdf...
Enviando el archivo a Azure AI Document Intelligence...
# A Tutorial on Principal Component Analysis

Jonathon Shlens*
Google Research
Mountain View, CA 94043

(Dated: April 7, 2014; Version 3.02)

Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
manuscript focuses on building a solid intuition for how and why principal component analysis works. This
manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics behind PCA. This
tutorial does not shy away from explaining the ideas informally, nor does it shy away from the mathematics. The
hope is that by addressing both aspects, readers of all levels will be able to gain a better understanding of PCA as
well as the when, the how and the why of applying this techni

In [5]:
def recursive_text_splitter(text: str, max_chunk_size: int = 500, overlap_size: int = 50) -> list[str]:
    """
    Divide un texto en segmentos (chunks) para RAG sin usar librerías externas.
    Respeta la estructura semántica (párrafos, oraciones, palabras) y añade solapamiento.
    
    :param text: El string de texto completo a dividir.
    :param max_chunk_size: Límite máximo de caracteres por segmento.
    :param overlap_size: Cantidad de caracteres a duplicar entre segmentos adyacentes.
    :return: Lista de strings (chunks).
    """
    # 1. Definimos la jerarquía de separadores (de mayor a menor peso semántico)
    separators = ["\n\n", "\n", ". ", " ", ""]
    
    def _split_recursive(chunk_text: str, current_seps: list[str]) -> list[str]:
        # Caso base 1: Si el fragmento ya entra en el tamaño máximo, se devuelve tal cual
        if len(chunk_text) <= max_chunk_size:
            return [chunk_text]
            
        # Caso base 2: Si nos quedamos sin separadores, cortamos estrictamente por caracteres
        if not current_seps:
            return [chunk_text[i:i + max_chunk_size] for i in range(0, len(chunk_text), max_chunk_size)]
            
        sep = current_seps[0]
        splits = chunk_text.split(sep)
        
        chunks = []
        current_chunk = ""
        
        for part in splits:
            # Reconstruimos el texto añadiendo el separador que eliminó el .split()
            potential_chunk = current_chunk + (sep if current_chunk else "") + part
            
            if len(potential_chunk) <= max_chunk_size:
                current_chunk = potential_chunk
            else:
                if current_chunk:
                    chunks.append(current_chunk)
                
                # Si una sola palabra/parte es más grande que el máximo, bajamos de nivel en los separadores
                if len(part) > max_chunk_size:
                    chunks.extend(_split_recursive(part, current_seps[1:]))
                    current_chunk = ""
                else:
                    current_chunk = part
                    
        if current_chunk:
            chunks.append(current_chunk)
            
        return chunks

    # Fase 1: Dividir el texto respetando la estructura semántica
    base_chunks = _split_recursive(text, separators)
    
    # Fase 2: Aplicar el solapamiento (Overlap)
    final_chunks = []
    for i, chunk in enumerate(base_chunks):
        if i == 0:
            final_chunks.append(chunk)
        else:
            prev_chunk = base_chunks[i - 1]
            # Tomamos los últimos 'overlap_size' caracteres del chunk anterior
            overlap_text = prev_chunk[-overlap_size:] if len(prev_chunk) >= overlap_size else prev_chunk
            
            # Construimos el nuevo chunk con el contexto previo
            combined_chunk = overlap_text + " " + chunk
            final_chunks.append(combined_chunk)
            
    return final_chunks



    

In [6]:
mis_chunks = recursive_text_splitter(text_pdf, max_chunk_size=150, overlap_size=30)
    
print(f"Total de chunks generados: {len(mis_chunks)}\n")
for idx, c in enumerate(mis_chunks):
    print(f"--- CHUNK {idx + 1} ({len(c)} caracteres) ---")
    print(c)
    print("-" * 30)

    

Total de chunks generados: 88

--- CHUNK 1 (140 caracteres) ---
# A Tutorial on Principal Component Analysis

Jonathon Shlens*
Google Research
Mountain View, CA 94043

(Dated: April 7, 2014; Version 3.02)
------------------------------
--- CHUNK 2 (137 caracteres) ---
: April 7, 2014; Version 3.02) Principal component analysis (PCA) is a mainstay of modern data analysis - a black box that is widely used
------------------------------
--- CHUNK 3 (139 caracteres) ---
 black box that is widely used but (sometimes) poorly understood. The goal of this paper is to dispel the magic behind this black box. This
------------------------------
--- CHUNK 4 (136 caracteres) ---
ic behind this black box. This manuscript focuses on building a solid intuition for how and why principal component analysis works. This
------------------------------
--- CHUNK 5 (138 caracteres) ---
component analysis works. This manuscript crystallizes this knowledge by deriving from simple intuitions, the mathematics be

In [ ]:
def get_embeddings_batch(text_list: list[str], api_key: str) -> list[list[float]]:
    """
    Recibe una lista de strings y devuelve una lista de vectores embeddings.
    Hace una sola petición eficiente a la API.
    """
    url = "https://api.openai.com/v1/embeddings"
    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Bearer {api_key}"
    }
    data = {
        "model": "text-embedding-3-small",
        "input": text_list  # <-- Aquí pasamos la lista completa
    }
    
    response = requests.post(url, json=data, headers=headers)
    
    if response.status_code == 200:
        res_json = response.json()
        # La API devuelve los datos ordenados en una lista. Los extraemos manteniendo el orden.
        # sorted asegura que si por alguna razón vienen desordenados, se alineen con tu lista original.
        sorted_data = sorted(res_json['data'], key=lambda x: x['index'])
        return [item['embedding'] for item in sorted_data]
    else:
        raise Exception(f"Error en la API de OpenAI: {response.text}")